# Retail Pricing Optimization with Business Constraints

## Business use case

Retail pricing is rarely a pure prediction problem. A useful decision system must translate a demand model into a price schedule while respecting the price points and promotion rules that the business can actually execute.

## Objective

This notebook uses a demand equation with current price, two lagged prices, and seasonal effects to optimize revenue over weeks 157–169. Four progressively more realistic Gurobi models show how unconstrained optimization changes when a price ladder and promotion policies are introduced.

## Modeling perspective

The project is intentionally decision-oriented: the predictive demand equation is treated as an input to optimization. The conclusions focus on the formulation and the business-rule progression.


## Step 1 — Prepare the optimization environment

NumPy and Pandas handle the planning data, Gurobi solves the optimization models, and Plotly is used to visualize the resulting weekly price schedules.


In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 57.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import plotly.express as px

## Step 2 — Define the planning horizon and demand model

The horizon covers 13 weeks. Demand depends on current price, the previous two prices, and a seasonal adjustment. The negative current-price coefficient captures the expected inverse relationship between price and demand, while lag terms allow recent pricing to influence the next weeks.


In [ ]:
# The range function range(a,b) creates a range of integers starting at <a> but ending at <b-1>
weeks = list(range(157,170))
print("weeks (t) ", weeks)
# Also note here that the first element of a list has index 0. The second element has index 1, and so on...
w1 = weeks[0] #denotes the number of the first week in our planning horizon
print(w1)
w2 = weeks[1] #denotes the number of the second week in our planning horizon
print(w2)

weeks (t)  [157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169]
157
158


In [ ]:
# From original price / sales(demand) data
# Developed linear demand model: used 27-156 weeks (3 years) of data; 27-104( years 1/2)Train, 105-156(last year)Test

# Variable: 𝑆𝑒𝑎𝑠𝑜𝑛X. Year divided int 13 seasons / 4 weeks per season (categorical variable)
# Variable: Price,  at time 𝑝_𝑡, 𝑝_(𝑡−1), 𝑝_(𝑡−2)
# Linear regression developed demand model (Constrain 1)
# Demand is characterized by our linearly additive model
# model: 𝑑_𝑡 = 2181 − 2801*𝑝_𝑡 + 929*𝑝_(𝑡−1) + 728*𝑝_(𝑡−2) −555.430*𝑆𝑒𝑎𝑠𝑜𝑛2 +...+ 949.056*𝑆𝑒𝑎𝑠𝑜𝑛13

# Store model parameters
intercept = 2181
# causal factors
p_coeff = -2801
p1_coeff = 929
p2_coeff = 728
# time series factor
season_coeff = {1: 0, 2: -555.430, 3: 81.746,  4: -406.774, 5: -26.122, 6: -8.292,
               7: -39.334, 8: -81.407, 9: 148.728, 10: 1107.254, 11: 1125.259, 12: 1171.240, 13: 949.056}

In [ ]:
# Here we create a dictionary that associates a season with each week in the planning horizon
season = {}
for w in weeks:
    season[w] = np.ceil((w % 52) / 4)

print(season)

{157: np.float64(1.0), 158: np.float64(1.0), 159: np.float64(1.0), 160: np.float64(1.0), 161: np.float64(2.0), 162: np.float64(2.0), 163: np.float64(2.0), 164: np.float64(2.0), 165: np.float64(3.0), 166: np.float64(3.0), 167: np.float64(3.0), 168: np.float64(3.0), 169: np.float64(4.0)}


## Step 3 — Solve an unconstrained pricing benchmark

The first model allows a continuous price up to 1.0 and maximizes total revenue across the horizon. This provides an economic benchmark before operational rules are imposed.


In [ ]:
# Create Gurobi model object - repository for all objects to be used in the model
mod1 = gp.Model ("price_model_1")

Restricted license - for non-production use only - expires 2027-11-29


In [ ]:
# Define decision variables
p = mod1.addVars(weeks, ub = 1)
print(p)

{157: <gurobi.Var *Awaiting Model Update*>, 158: <gurobi.Var *Awaiting Model Update*>, 159: <gurobi.Var *Awaiting Model Update*>, 160: <gurobi.Var *Awaiting Model Update*>, 161: <gurobi.Var *Awaiting Model Update*>, 162: <gurobi.Var *Awaiting Model Update*>, 163: <gurobi.Var *Awaiting Model Update*>, 164: <gurobi.Var *Awaiting Model Update*>, 165: <gurobi.Var *Awaiting Model Update*>, 166: <gurobi.Var *Awaiting Model Update*>, 167: <gurobi.Var *Awaiting Model Update*>, 168: <gurobi.Var *Awaiting Model Update*>, 169: <gurobi.Var *Awaiting Model Update*>}


In [ ]:
# Set objective function
# First 2 lines fully written out then use a short cut for writing out each line
# After the first 2 weeks, the remaining weeks[2:], are developed using the sum() and a for loop  (weeks 159 on...)
# price at week(wx) * demand (from linear demand fucntion at weekx)
obj_fn = mod1.setObjective(p[w1] * (intercept + p_coeff*p[w1] + p1_coeff*1 + p2_coeff*1 + season_coeff[season[w1]]) +
                           p[w2] * (intercept + p_coeff*p[w2] + p1_coeff*p[w1] + p2_coeff*1 + season_coeff[season[w2]]) +
                           sum(p[w] * (intercept + p_coeff*p[w] + p1_coeff*p[w-1] + p2_coeff*p[w-2] + season_coeff[season[w]]) for w in weeks[2:]),
                          GRB.MAXIMIZE)

In [ ]:
mod1.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 0 rows, 13 columns and 0 nonzeros (Max)
Model fingerprint: 0x7ae80c7d
Model has 13 linear objective coefficients
Model has 36 quadratic objective terms
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [2e+03, 4e+03]
  QObjective range [1e+03, 6e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [0e+00, 0e+00]

Presolve time: 0.01s
Presolved: 0 rows, 13 columns, 0 nonzeros
Presolved model has 36 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 12
 AA' NZ     : 6.600e+01
 Factor NZ  : 7.800e+01
 Factor Ops : 6.500e+02 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl  

In [ ]:
p[157] # to see 1st prediction

<gurobi.Var C0 (value 0.9621363051399713)>

In [ ]:
p[158] # to see 2nd prediction

<gurobi.Var C1 (value 0.9480218163490649)>

In [ ]:
# to see all predictions and save to a dataframe
season = {}
for w in weeks:
    week_in_year = ((w - 1) % 52) + 1  # wrap into 1–52
    season[w] = int(np.ceil(week_in_year / 4))

print(season)


{157: 1, 158: 1, 159: 1, 160: 1, 161: 2, 162: 2, 163: 2, 164: 2, 165: 3, 166: 3, 167: 3, 168: 3, 169: 4}


In [ ]:
df1 = pd.DataFrame(data = None, index = weeks, columns = ["price"])
for w in weeks:
    df1.loc[w,"price"] = p[w].x

fig = px.line(df1, x=df1.index, y='price', markers=True)
fig.update_layout(plot_bgcolor= "white", xaxis_title= "week")
fig.update_traces(line_color= "red")
fig.show()

## Step 4 — Restrict decisions to an executable price ladder

The second model limits each week to one of four price points: 1.0, 0.9, 0.8, or 0.7. Binary variables connect the selected ladder level to the weekly price, converting the problem into a mixed-integer formulation.


In [ ]:
p_ladder = [1.0, 0.9, 0.8, 0.7] #create a list for price ladder

In [ ]:
# Create model object
mod2 = gp.Model ("price_model_2")

In [ ]:
# Define decision variables which includes ladder constraint
p = mod2.addVars(weeks)
x = mod2.addVars(weeks, p_ladder, vtype= GRB.BINARY)

In [ ]:
# Set objective function
obj_fn = mod2.setObjective(p[w1] * (intercept + p_coeff*p[w1] + p1_coeff*1 + p2_coeff*1 + season_coeff[season[w1]]) +
                           p[w2] * (intercept + p_coeff*p[w2] + p1_coeff*p[w1] + p2_coeff*1 + season_coeff[season[w2]]) +
                           sum(p[w] * (intercept + p_coeff*p[w] + p1_coeff*p[w-1] + p2_coeff*p[w-2] + season_coeff[season[w]]) for w in weeks[2:]),
                          GRB.MAXIMIZE)

In [ ]:
# Select price ladder value
constr_select_ladder = mod2.addConstrs(sum(x[w,k] for k in p_ladder) == 1 for w in weeks)
# Select price
constr_select_price = mod2.addConstrs(p[w] == sum(k * x[w,k] for k in p_ladder) for w in weeks)

In [ ]:
mod2.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 26 rows, 65 columns and 117 nonzeros (Max)
Model fingerprint: 0x5eb5a9f6
Model has 13 linear objective coefficients
Model has 36 quadratic objective terms
Variable types: 13 continuous, 52 integer (52 binary)
Coefficient statistics:
  Matrix range     [7e-01, 1e+00]
  Objective range  [2e+03, 4e+03]
  QObjective range [1e+03, 6e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 11514.818400
Presolve removed 0 rows and 13 columns
Presolve time: 0.00s
Presolved: 26 rows, 52 columns, 91 nonzeros
Presolved model has 36 quadratic objective terms
Variable types: 13 continuous, 39 integer (39 binary)

Root relaxation: objective 1.201770e+04, 37 iterations, 0.00 seconds (0.00 work units)

    Nodes  

In [ ]:
df2 = pd.DataFrame(data = None, index = weeks, columns = ["price"])
for w in weeks:
    df2.loc[w,"price"] = p[w].x
df2

,price
157,1.0
158,1.0
159,0.9
160,0.9
161,0.8
162,0.8
163,0.8
164,0.8
165,0.9
166,0.9


In [ ]:
fig = px.line(df2, x=df2.index, y='price', markers=True)
fig.update_layout(plot_bgcolor= "white", xaxis_title= "week")
fig.update_traces(line_color= "red")
fig.show()

## Step 5 — Limit promotional frequency

The third model keeps the price ladder and adds a cap of four promotional weeks. This introduces a realistic trade-off between short-term revenue optimization and promotion discipline.


In [ ]:
# Create model object
mod3 = gp.Model ("price_model_3")

In [ ]:
# Define decision variables
p = mod3.addVars(weeks)
x = mod3.addVars(weeks, p_ladder, vtype= GRB.BINARY)

In [ ]:
# Set objective function
obj_fn = mod3.setObjective(p[w1] * (intercept + p_coeff*p[w1] + p1_coeff*1 + p2_coeff*1 + season_coeff[season[w1]]) +
                           p[w2] * (intercept + p_coeff*p[w2] + p1_coeff*p[w1] + p2_coeff*1 + season_coeff[season[w2]]) +
                           sum(p[w] * (intercept + p_coeff*p[w] + p1_coeff*p[w-1] + p2_coeff*p[w-2] + season_coeff[season[w]]) for w in weeks[2:]),
                          GRB.MAXIMIZE)

In [ ]:
# Select price ladder value
constr_select_ladder = mod3.addConstrs(sum(x[w,k] for k in p_ladder) == 1 for w in weeks)
# Select price
constr_select_price = mod3.addConstrs(p[w] == sum(k * x[w,k] for k in p_ladder) for w in weeks)
# At most 4 promotions
constr_4_promo = mod3.addConstr(sum(x[w,k] for k in p_ladder[1:] for w in weeks) <= 4)

In [ ]:
mod3.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 27 rows, 65 columns and 156 nonzeros (Max)
Model fingerprint: 0x8356128d
Model has 13 linear objective coefficients
Model has 36 quadratic objective terms
Variable types: 13 continuous, 52 integer (52 binary)
Coefficient statistics:
  Matrix range     [7e-01, 1e+00]
  Objective range  [2e+03, 4e+03]
  QObjective range [1e+03, 6e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Found heuristic solution: objective 10890.440000
Presolve time: 0.00s
Presolved: 27 rows, 65 columns, 143 nonzeros
Presolved model has 36 quadratic objective terms
Variable types: 13 continuous, 52 integer (52 binary)

Root relaxation: objective 1.194927e+04, 65 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective

In [ ]:
df3 = pd.DataFrame(data = None, index = weeks, columns = ["price"])
for w in weeks:
    df3.loc[w,"price"] = p[w].x
df3

,price
157,1.0
158,1.0
159,1.0
160,1.0
161,1.0
162,0.9
163,0.9
164,1.0
165,1.0
166,1.0


In [ ]:
fig = px.line(df3, x=df3.index, y='price', markers=True)
fig.update_layout(plot_bgcolor= "white", xaxis_title= "week")
fig.update_traces(line_color= "red")
fig.show()

## Step 6 — Add the full promotion policy

The final model also prevents consecutive promotional weeks. This produces a price schedule that is not only optimized against modeled demand, but also aligned with a rule a merchandising team could plausibly execute.


In [ ]:
# Create model object
mod4 = gp.Model ("price_model_4")

In [ ]:
# Define decision variables
p = mod4.addVars(weeks)
x = mod4.addVars(weeks, p_ladder, vtype= GRB.BINARY)

In [ ]:
# Set objective function
obj_fn = mod4.setObjective(p[w1] * (intercept + p_coeff*p[w1] + p1_coeff*1 + p2_coeff*1 + season_coeff[season[w1]]) +
                           p[w2] * (intercept + p_coeff*p[w2] + p1_coeff*p[w1] + p2_coeff*1 + season_coeff[season[w2]]) +
                           sum(p[w] * (intercept + p_coeff*p[w] + p1_coeff*p[w-1] + p2_coeff*p[w-2] + season_coeff[season[w]]) for w in weeks[2:]),
                          GRB.MAXIMIZE)

In [ ]:
# Select price ladder value
constr_select_ladder = mod4.addConstrs(sum(x[w,k] for k in p_ladder) == 1 for w in weeks)
# Select price
constr_select_price = mod4.addConstrs(p[w] == sum(k * x[w,k] for k in p_ladder) for w in weeks)
# At most 4 promotions
constr_4_promo = mod4.addConstr(sum(x[w,k] for k in p_ladder[1:] for w in weeks) <= 4)
# No consecutive promotions
constr_no_consec_promo = mod4.addConstrs((sum(x[w,k] for k in p_ladder[1:]) + sum(x[w+1,k] for k in p_ladder[1:]) <= 1) for w in weeks[:-1])

In [ ]:
mod4.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 39 rows, 65 columns and 228 nonzeros (Max)
Model fingerprint: 0xf7b2a2b1
Model has 13 linear objective coefficients
Model has 36 quadratic objective terms
Variable types: 13 continuous, 52 integer (52 binary)
Coefficient statistics:
  Matrix range     [7e-01, 1e+00]
  Objective range  [2e+03, 4e+03]
  QObjective range [1e+03, 6e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Found heuristic solution: objective 11126.462000
Presolve time: 0.00s
Presolved: 39 rows, 65 columns, 215 nonzeros
Presolved model has 36 quadratic objective terms
Variable types: 13 continuous, 52 integer (52 binary)

Root relaxation: objective 1.187440e+04, 73 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective

In [ ]:
df4 = pd.DataFrame(data = None, index = weeks, columns = ["price"])
for w in weeks:
    df4.loc[w,"price"] = p[w].x
df4

,price
157,1.0
158,1.0
159,1.0
160,1.0
161,1.0
162,0.9
163,1.0
164,0.9
165,1.0
166,1.0


In [ ]:
fig = px.line(df4, x=df4.index, y='price', markers=True)
fig.update_layout(plot_bgcolor= "white", xaxis_title= "week")
fig.update_traces(line_color= "red")
fig.show()

## Technical conclusions

The notebook demonstrates an important modeling pattern: first separate demand estimation from decision optimization, then add business constraints incrementally. The shift from a continuous model to mixed-integer price-ladder models is what makes the result operationally meaningful.

## Business conclusions

The model structure lets a pricing team quantify the cost of policy constraints instead of debating them qualitatively. It can answer questions such as how much revenue is traded for a fixed price ladder, a promotion cap, or a no-back-to-back-promotion rule.

## Limitations and next steps

The demand equation is treated as fixed and deterministic. A production system should validate elasticity over time, enforce nonnegative demand, account for margin and inventory rather than revenue alone, and test uncertainty in the demand coefficients.
